# 01 — Eksplorasi Data

Memeriksa kualitas data, distribusi gol, dan menghitung **Elo rating** seluruh tim secara kronologis dari 49 ribu+ pertandingan internasional (1872–2026).

Dataset yang dipakai di proyek ini dan perannya:

| Dataset | Peran |
|---|---|
| `results.csv` | Training utama (49 ribu laga) + 72 fixture grup 2026 |
| `shootouts.csv` | Model adu penalti babak knockout |
| `goalscorers.csv` | Eksplorasi pencetak gol & profil tim |
| `former_names.csv` | Validasi kontinuitas nama negara (sudah ter-apply di results.csv) |
| `wc_2026_groups.csv` | 48 peserta, grup A–L, nilai pasar skuad, pemain kunci |
| `wc_2026_teams_snapshot.csv`, `wc_coaches_2026.csv`, `wc_team_alltime_stats.csv` | Profil tim di dashboard |
| `fifa_ranking_2026-06-08.csv` | Validasi silang Elo vs ranking FIFA |
| `matches_1930_2022.csv` | Verifikasi duplikasi (subset results.csv) |

Fitur training hanya dari `results.csv` (Elo, form, h2h) — data snapshot 2026 (nilai pasar, ranking) **tidak** dipakai sebagai fitur model karena tidak tersedia historis (anti-leakage), tapi dipakai untuk validasi silang & tampilan.

In [1]:
import sys
from pathlib import Path

BASE = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(BASE / 'src'))

import numpy as np
import pandas as pd
import plotly.express as px

from data_prep import (load_results, load_fixtures_2026, load_groups, load_fifa_ranking,
                       load_goalscorers, compute_elo, TOURNAMENT_START, WC_MATCHES_CSV)

results = load_results()
print(f'Total pertandingan dimainkan : {len(results):,}')
print(f'Rentang tanggal              : {results.date.min().date()} s/d {results.date.max().date()}')
print(f'Jumlah tim unik              : {pd.concat([results.home_team, results.away_team]).nunique()}')

Total pertandingan dimainkan : 49,405
Rentang tanggal              : 1872-11-30 s/d 2026-06-10
Jumlah tim unik              : 336


## Validasi normalisasi nama tim & fixture 2026
Semua 48 peserta dan 72 laga grup harus ter-resolve ke nama kanonik (assert di dalam loader).

In [2]:
groups = load_groups()
fixtures = load_fixtures_2026()
hist_teams = set(results.home_team) | set(results.away_team)
missing = set(groups.team) - hist_teams
assert not missing, f'Tim tanpa riwayat: {missing}'
print('OK: 48 tim peserta semuanya punya riwayat pertandingan.')
print('OK: 72 fixture fase grup ter-load dengan grup konsisten.')
groups.groupby('group')['team'].apply(list).to_frame('tim')

OK: 48 tim peserta semuanya punya riwayat pertandingan.
OK: 72 fixture fase grup ter-load dengan grup konsisten.


,tim
group,
A,"[Mexico, South Korea, South Africa, Czech Repu..."
B,"[Canada, Switzerland, Qatar, Bosnia and Herzeg..."
C,"[Brazil, Morocco, Scotland, Haiti]"
D,"[United States, Paraguay, Australia, Turkey]"
E,"[Germany, Curaçao, Ivory Coast, Ecuador]"
F,"[Netherlands, Japan, Sweden, Tunisia]"
G,"[Belgium, Egypt, Iran, New Zealand]"
H,"[Spain, Cape Verde, Saudi Arabia, Uruguay]"
I,"[France, Senegal, Iraq, Norway]"


## Validasi `former_names.csv` & duplikasi antar dataset
1. `results.csv` seharusnya sudah memakai nama negara saat ini (Zaire→DR Congo, Soviet Union→Russia, dst.) sehingga riwayat Elo kontinu.
2. `matches_1930_2022.csv` adalah subset dari `results.csv` — laga Piala Dunia sudah tercakup, jadi tidak dipakai terpisah (hindari data dobel).

In [3]:
former = ['Zaïre', 'Zaire', 'Soviet Union', 'Burma', 'Upper Volta', 'Netherlands Antilles', 'Macedonia']
leftover = [n for n in former if n in hist_teams]
assert not leftover, f'Nama lama masih ada: {leftover}'
print('OK: nama-nama lama (Zaire, Soviet Union, dll.) sudah dipetakan ke nama saat ini di results.csv')

wc_sub = pd.read_csv(WC_MATCHES_CSV)
wc_in_results = results[results.tournament == 'FIFA World Cup']
print(f'Laga PD di matches_1930_2022.csv : {len(wc_sub):,}')
print(f'Laga PD di results.csv           : {len(wc_in_results):,}')
print('=> results.csv mencakup seluruh laga Piala Dunia; file terpisah tidak perlu dipakai (duplikat).')

OK: nama-nama lama (Zaire, Soviet Union, dll.) sudah dipetakan ke nama saat ini di results.csv
Laga PD di matches_1930_2022.csv : 964
Laga PD di results.csv           : 964
=> results.csv mencakup seluruh laga Piala Dunia; file terpisah tidak perlu dipakai (duplikat).


## Distribusi gol
Distribusi gol mendekati Poisson — dasar pemilihan objective `count:poisson` untuk model skor.

In [4]:
modern = results[results.date >= '1990-01-01']
goals = pd.concat([modern.home_score, modern.away_score])
dist = goals.value_counts(normalize=True).sort_index().loc[:8]
lam = goals.mean()
from math import exp, factorial
poisson_teo = [exp(-lam) * lam**k / factorial(k) for k in dist.index]
cmp_df = pd.DataFrame({'Data aktual': dist.values, 'Poisson teoritis': poisson_teo}, index=dist.index)
fig = px.bar(cmp_df, barmode='group', title=f'Distribusi gol per tim per laga (sejak 1990, rata-rata λ={lam:.2f})',
             labels={'index': 'Jumlah gol', 'value': 'Proporsi', 'variable': ''})
fig.show()

## Elo rating — kondisi terkini (menjelang kickoff 11 Juni 2026)

In [5]:
pre_wc = results[results.date < TOURNAMENT_START]
_, elo_final = compute_elo(pre_wc)
participants = set(groups.team)
elo_df = (pd.Series(elo_final).loc[lambda s: s.index.isin(participants)]
          .sort_values(ascending=False).round(0).astype(int).to_frame('Elo'))
fig = px.bar(elo_df.head(20).iloc[::-1], orientation='h', title='Top 20 Elo peserta Piala Dunia 2026',
             labels={'index': '', 'value': 'Elo'}).update_layout(showlegend=False)
fig.show()
elo_df.head(10)

,Elo
Spain,2219
Argentina,2190
France,2125
England,2091
Brazil,2069
Colombia,2064
Portugal,2046
Ecuador,2028
Netherlands,2011
Germany,2005


## Validasi silang: Elo internal vs ranking FIFA vs nilai pasar skuad
Tiga ukuran kekuatan yang independen seharusnya berkorelasi kuat. Jika Elo kita sejalan dengan poin FIFA dan nilai pasar (sumber sama sekali berbeda), itu bukti Elo engine bekerja benar — tanpa memasukkan keduanya sebagai fitur model.

In [6]:
rank = load_fifa_ranking()[['team', 'rank', 'points']]
mv = groups[['team', 'squad_market_value_eur_millions', 'key_player']]
cross = (elo_df.reset_index(names='team').merge(rank, on='team').merge(mv, on='team'))
corr_fifa = cross.Elo.corr(cross.points)
corr_mv = cross.Elo.corr(np.log(cross.squad_market_value_eur_millions))
print(f'Korelasi Elo vs poin FIFA          : {corr_fifa:.3f}')
print(f'Korelasi Elo vs log(nilai pasar)   : {corr_mv:.3f}')
fig = px.scatter(cross, x='points', y='Elo', size='squad_market_value_eur_millions',
                 hover_name='team', title='Elo internal vs poin FIFA (ukuran bubble = nilai pasar skuad)',
                 labels={'points': 'Poin ranking FIFA (Jun 2026)'})
fig.show()

Korelasi Elo vs poin FIFA          : 0.911
Korelasi Elo vs log(nilai pasar)   : 0.817


## Evolusi Elo tim-tim besar sejak 2000

In [7]:
from data_prep import elo_update, INITIAL_ELO
watch = ['Brazil', 'Argentina', 'France', 'Spain', 'England', 'Germany']
ratings, history = {}, []
for row in pre_wc.itertuples(index=False):
    eh = ratings.get(row.home_team, INITIAL_ELO)
    ea = ratings.get(row.away_team, INITIAL_ELO)
    ratings[row.home_team], ratings[row.away_team] = elo_update(eh, ea, row.home_score, row.away_score, row.tournament, bool(row.neutral))
    if row.date.year >= 2000:
        for t in (row.home_team, row.away_team):
            if t in watch:
                history.append((row.date, t, ratings[t]))
hist_df = pd.DataFrame(history, columns=['tanggal', 'tim', 'elo'])
fig = px.line(hist_df, x='tanggal', y='elo', color='tim', title='Evolusi Elo 2000–2026')
fig.show()

## Pencetak gol internasional (goalscorers.csv)
Top skor sepanjang masa + pencetak gol teraktif di antara peserta 2026.

In [8]:
gs = load_goalscorers()
gs_valid = gs[(~gs.own_goal.fillna(False)) & gs.scorer.notna()]
top = gs_valid.groupby(['scorer', 'team']).size().sort_values(ascending=False).head(15)
top_df = top.reset_index(name='gol')
fig = px.bar(top_df.iloc[::-1], x='gol', y='scorer', orientation='h', color='team',
             title='Top 15 pencetak gol internasional sepanjang masa')
fig.show()
recent = gs_valid[gs_valid.date >= '2022-01-01']
top_recent = recent[recent.team.isin(participants)].groupby(['scorer', 'team']).size().sort_values(ascending=False).head(10)
print('Top 10 pencetak gol peserta 2026 (sejak 2022):')
print(top_recent.to_string())

Top 10 pencetak gol peserta 2026 (sejak 2022):
scorer             team       
Erling Haaland     Norway         35
Kylian Mbappé      France         27
Harry Kane         England        26
Cristiano Ronaldo  Portugal       26
Cody Gakpo         Netherlands    18
Romelu Lukaku      Belgium        18
Mehdi Taremi       Iran           17
Viktor Gyökeres    Sweden         17
Bruno Fernandes    Portugal       17
Lionel Messi       Argentina      17
